In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("source_file_format", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
base_landing_path = "/Volumes/adb_caresync_preprod/landing/landing_volume"
source_path = f"{base_landing_path}/{source_system}"
table_path = f"{source_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}/"


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


missing_path = next((path for path in [source_path, table_path, landing_path] if not path_exists(path)), None)

if missing_path:
    message = (
        f"No new files are received for source_system={source_system}, "
        f"table_name={table_name}, landing_timestamp={landing_timestamp}."
    )
    print(message)
    dbutils.notebook.exit(message)

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

hospital_id,hospital_name,city,state,hospital_type,bed_count,created_at,updated_at
1,Austin General Hospital,Austin,TX,General,436,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
2,Dallas Medical Center,Dallas,TX,Specialty,242,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
3,Houston Regional Hospital,Houston,TX,Teaching,135,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
4,San Antonio Community Hospital,San Antonio,TX,Community,383,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
5,Fort Worth Health Center,Fort Worth,TX,Regional,467,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
6,Phoenix Care Center,Phoenix,AZ,General,463,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
7,Denver Memorial Hospital,Denver,CO,Specialty,260,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
8,Seattle General Hospital,Seattle,WA,Teaching,125,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
9,Portland Medical Center,Portland,OR,Community,400,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z
10,Chicago Regional Hospital,Chicago,IL,Regional,164,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z


In [0]:
from pyspark.sql.functions import *
landing_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)).withColumn("insert_timestamp", lit(current_timestamp()))

In [0]:
landing_df.write.mode("append").saveAsTable(f"adb_caresync_preprod.bronze.{table_name}")

In [0]:
%sql
Select * from adb_caresync_preprod.bronze.hospitals

hospital_id,hospital_name,city,state,hospital_type,bed_count,created_at,updated_at,landing_timestamp,insert_timestamp
1,Austin General Hospital,Austin,TX,General,436,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
2,Dallas Medical Center,Dallas,TX,Specialty,242,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
3,Houston Regional Hospital,Houston,TX,Teaching,135,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
4,San Antonio Community Hospital,San Antonio,TX,Community,383,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
5,Fort Worth Health Center,Fort Worth,TX,Regional,467,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
6,Phoenix Care Center,Phoenix,AZ,General,463,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
7,Denver Memorial Hospital,Denver,CO,Specialty,260,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
8,Seattle General Hospital,Seattle,WA,Teaching,125,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
9,Portland Medical Center,Portland,OR,Community,400,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
10,Chicago Regional Hospital,Chicago,IL,Regional,164,2026-09-29T15:20:19.450998Z,2026-09-29T15:20:19.450998Z,2026-09-30_12:36:56,2026-09-30T14:54:16.593732Z
